# Information of the notebook
---

This notebook is prepared for the **Capstone BBO Stage 1** Project.

The aim of the Capstone BBO Stage 1 is to build the skills and habits required for the BBO challenge in Stage 2.

For this experiment, **House Prices - Advanced Regression Techniques, link from Kaggle**  is chosen from Kaggle.

(https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/overview Links to an external site.)

Ordinal variable treatment is conducted for **columns with NAN** values.  

**PCA** and **StandardScaler** are conducted for preprocessing and dimensionality reduction.

**Linear Regression** with **K-Fold Cross-Validation** is checked to understand the linear score.

**PolynomialFeatures** cross-validation is reviewed and noted poor validation score. 

The preliminary test from **XGBRegressor** CV_Score and R_Square are found to be the highest.

Differences between XGBRegressor Prediction and **Sample submission** are checked.


In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression

from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score

import time
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split as tts, cross_val_score as cv, KFold

pd.set_option('display.max_columns', None)

In [2]:
data = pd.read_csv('train.csv')

In [3]:
data.head(2)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2003,2003,Gable,CompShg,VinylSd,VinylSd,BrkFace,196.0,Gd,TA,PConc,Gd,TA,No,GLQ,706,Unf,0,150,856,GasA,Ex,Y,SBrkr,856,854,0,1710,1,0,2,1,3,1,Gd,8,Typ,0,NaN,Attchd,2003.0,RFn,2,548,TA,TA,Y,0,61,0,0,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,Gtl,Veenker,Feedr,Norm,1Fam,1Story,6,8,1976,1976,Gable,CompShg,MetalSd,MetalSd,NaN,0.0,TA,TA,CBlock,Gd,TA,Gd,ALQ,978,Unf,0,284,1262,GasA,Ex,Y,SBrkr,1262,0,0,1262,0,1,2,0,3,1,TA,6,Typ,1,TA,Attchd,1976.0,RFn,2,460,TA,TA,Y,298,0,0,0,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500


In [4]:
df = data.drop(columns = 'Id' )

In [5]:
print(f"\
\nNull Found:          {df.isnull().sum().sum()}\
\nDuplicate Found:     {df.duplicated().sum()}\
\nTotal Rows:          {df.shape[0]}\
\nTotal Columns:       {df.shape[1]}")


Null Found:          7829
Duplicate Found:     0
Total Rows:          1460
Total Columns:       80


# Numeric Columns

In [6]:
# Numeric Columns NA Treatments

num_data = df.select_dtypes(include = ['int', 'float'])
null_num_data = num_data.isnull().sum()
print('-'*39)
print('| Numeric Columns which has NAN Value |')
print('-'*39)
print(df[null_num_data[null_num_data>0].reset_index()['index'].tolist()].head())

---------------------------------------
| Numeric Columns which has NAN Value |
---------------------------------------
   LotFrontage  MasVnrArea  GarageYrBlt
0         65.0       196.0       2003.0
1         80.0         0.0       1976.0
2         68.0       162.0       2001.0
3         60.0         0.0       1998.0
4         84.0       350.0       2000.0


In [7]:
# NaN filled with zero
df['MasVnrArea'] = df['MasVnrArea'].fillna(0)

# NaN filled with -999 and created a column which indicate has garrage or not
import numpy as np
df['has_garage'] = np.where(df['GarageYrBlt'].isnull(), 1, 0)
df['GarageYrBlt'] = df['GarageYrBlt'].fillna(-999)

#NaN filled with mean of near by location of house  
group_mean = df.groupby('Street')['LotFrontage'].mean()
Map = df['Street'].map(group_mean)
df['LotFrontage'] = df['LotFrontage'].fillna(Map)

# Objects Columns

In [8]:
# Objects Columns NA Treatments

cat_data = df.drop(columns = num_data.columns)
null_cat_data = cat_data.isnull().sum()
print('-'*39)
print('| Objects Columns which has NAN Value |')
print('-'*39)
print(null_cat_data[null_cat_data>0].reset_index()['index'].tolist())

---------------------------------------
| Objects Columns which has NAN Value |
---------------------------------------
['Alley', 'MasVnrType', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Electrical', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature']


In [9]:
# Feature actually not exist will be filled with None
df[['Alley', 'MasVnrType', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
       'BsmtFinType1', 'BsmtFinType2', 'FireplaceQu',
       'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'PoolQC',
       'Fence', 'MiscFeature']] = df[['Alley', 'MasVnrType', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
       'BsmtFinType1', 'BsmtFinType2', 'FireplaceQu',
       'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'PoolQC',
       'Fence', 'MiscFeature']].fillna('None')

#NaN will be filled with Mode (Most frequent Comming item)
mode = df['Electrical'].mode()[0]
df['Electrical'] = df['Electrical'].fillna(mode)

In [10]:
print(f"\
\nNull Found:          {df.isnull().sum().sum()}\
\nDuplicate Found:     {df.duplicated().sum()}\
\nTotal Rows:          {df.shape[0]}\
\nTotal Columns:       {df.shape[1]}")


Null Found:          0
Duplicate Found:     0
Total Rows:          1460
Total Columns:       81


## Ordinal Variables Treatment

In [11]:
Quality_Col = df[['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC', 
    'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC']].columns.tolist()
Map_QC = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}
for i in Quality_Col:
    df[i] = df[i].map(Map_QC)

Map_BM = {'GLQ': 6, 'ALQ': 5, 'BLQ': 4, 'Rec': 3, 'LwQ': 2, 'Unf': 1, 'None': 0}
df['BsmtFinType1'] = df['BsmtFinType1'].map(Map_BM)
df['BsmtFinType2'] = df['BsmtFinType2'].map(Map_BM)

Map_BME = {'Gd': 4, 'Av': 3, 'Mn': 2, 'No': 1, 'None': 0}
df['BsmtExposure'] = df['BsmtExposure'].map(Map_BME)

Map_GF = {'Fin': 3, 'RFn': 2, 'Unf': 1, 'None':0}
df['GarageFinish'] = df['GarageFinish'].map(Map_GF)

Map_LS = {'Gtl': 2, 'Mod': 1, 'Sev': 0}
df['LandSlope'] = df['LandSlope'].map(Map_LS)

Map_LtS = {'Reg': 3, 'IR1': 2, 'IR2': 1, 'IR3': 0}
df['LotShape'] = df['LotShape'].map(Map_LtS)

from sklearn.preprocessing import LabelEncoder
cat_col = df.select_dtypes(exclude = ['int', 'float']).astype('category').columns.tolist()
for i in cat_col:
    df[i] = LabelEncoder().fit_transform(np.array(df[i]))

In [12]:
corr_ = df.select_dtypes(include = ['int', 'float']).corr()['SalePrice'].reset_index()
Mod = df.select_dtypes(include = ['int', 'float']).corr()['SalePrice'].abs().reset_index()
Top = corr_.merge(Mod, on = 'index').sort_values(by = 'SalePrice_y', ascending = False)
print('-'*33)
print('| Top 5 Correlation with Target |')
print('-'*33)
print(Top.iloc[1:6, :-1])

---------------------------------
| Top 5 Correlation with Target |
---------------------------------
          index  SalePrice_x
16  OverallQual     0.790982
45    GrLivArea     0.708624
26    ExterQual     0.682639
52  KitchenQual     0.659600
60   GarageCars     0.640409


In [13]:
#  Multicollinearity Pairs

melt = df.corr().reset_index().melt(id_vars='index').merge(corr_, on='index')
melt = melt[(melt['index'] != melt['variable']) & ((melt['value'] > 0.9) | (melt['value'] < -0.9))]
print('-'*51)
print('|    Multicollinearity Pairs (|r| > 0.9)       |')
print('-'*51)
print(melt if melt['index'].count() >0  else 'None found ✅')
print()

---------------------------------------------------
|    Multicollinearity Pairs (|r| > 0.9)       |
---------------------------------------------------
            index     variable     value  SalePrice
4760   GarageQual  GarageYrBlt  0.945539   0.273839
4761   GarageCond  GarageYrBlt  0.948646   0.263191
4778   has_garage  GarageYrBlt -0.999381  -0.236832
5080  GarageYrBlt   GarageQual  0.945539   0.253221
5085   GarageCond   GarageQual  0.959172   0.263191
5102   has_garage   GarageQual -0.942499  -0.236832
5161  GarageYrBlt   GarageCond  0.948646   0.253221
5165   GarageQual   GarageCond  0.959172   0.273839
5183   has_garage   GarageCond -0.946245  -0.236832
5741       PoolQC     PoolArea  0.937057   0.111696
5821     PoolArea       PoolQC  0.937057   0.092404
6538  GarageYrBlt   has_garage -0.999381   0.253221
6542   GarageQual   has_garage -0.942499   0.273839
6543   GarageCond   has_garage -0.946245   0.263191



# PCA and StandardScaler

In [14]:
garage_cols = df[['GarageYrBlt', 'GarageQual', 'GarageCond', 'has_garage']]

SG = StandardScaler().fit_transform(garage_cols)
pca = PCA(n_components = 1)
pca.fit(SG)
print(pca.explained_variance_ratio_)

[0.96771257]


In [15]:
df['pca_garage'] = pca.fit_transform(SG)
df.drop(columns=garage_cols.columns, inplace=True)
df.drop(columns=['PoolArea'], inplace=True)

In [16]:
print(f"\
\nNull Found:          {df.isnull().sum().sum()}\
\nDuplicate Found:     {df.duplicated().sum()}\
\nTotal Rows:          {df.shape[0]}\
\nTotal Columns:       {df.shape[1]}")


Null Found:          0
Duplicate Found:     0
Total Rows:          1460
Total Columns:       77


# define x and y

In [17]:
y = df[['SalePrice']]
x = df.drop(columns = 'SalePrice')

In [18]:
x_train, x_test, y_train, y_test = tts(x, y, test_size = 0.20, random_state = 42)

kf = KFold(n_splits = 5, shuffle = True, random_state = 42)

# Combining Linear Regression with K-Fold Cross-Validation 

In [19]:
# Combining Linear Regression with K-Fold Cross-Validation 

lm = LinearRegression()

lm_score = cv(lm, x_train, y_train, cv = kf)
lm_score.mean()

np.float64(0.7767286121195969)

<div class="alert-info">
0.77 indicates a moderately predictive model that explains 77% of the variance in your target.
</div>

# PolynomialFeatures cross validation

In [20]:
# poly regression

d = [2,3]
for i in d:
    pipe = Pipeline([
        ('feature', PolynomialFeatures(degree = i)),
        ('model', lm)
         ])
    poly_score = cv(pipe, x_train, y_train, cv = kf)
    print(f"degree = {i} CV_Score: {poly_score.mean()}")

degree = 2 CV_Score: 0.1301260565466813
degree = 3 CV_Score: -76.02854604399955


<div class="alert-warning">
A cross-validation score (CV_Score) of 0.13 indicates that your model is performing very poorly.
</div>

# DecisionTreeRegressor

In [21]:

depth   = [2, 3, 4, 5, 7, None]
options = ['squared_error', 'friedman_mse', 'absolute_error', 'poisson']

print(f"{'Depth':<6} | {'Criterion':<15} | {'CV Score':<10} | {'R²'}")
print('-' * 50)

results = []
for d in depth:
    for o in options:
        dt       = DecisionTreeRegressor(criterion=o, max_depth=d, random_state=42)
        dt.fit(x_train, y_train)
        y_hat    = dt.predict(x_test)
        cv_score = cv(dt, x_train, y_train, cv=kf).mean()
        r2       = r2_score(y_test, y_hat)
        results.append({'Depth': d, 'Criterion': o, 'CV Score': cv_score, 'R²': r2})
        print(f'{str(d):<6} | {o:<15} | {cv_score:.4f}     | {r2:.4f}')

results_df = pd.DataFrame(results).sort_values('CV Score', ascending=False)
print('\nBest combination by CV Score:')
print(results_df.iloc[0])

Depth  | Criterion       | CV Score   | R²
--------------------------------------------------
2      | squared_error   | 0.5654     | 0.6645


C:\Users\May Zune\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\tree\_classes.py:1351: FutureWarning: Value `"friedman_mse"` for `criterion` is deprecated and will be removed in 1.11. It maps to `"squared_error"` as both were always equivalent. Use `criterion="squared_error"` to remove this warning.
  warnings.warn(


2      | friedman_mse    | 0.5654     | 0.6645
2      | absolute_error  | 0.5888     | 0.5812
2      | poisson         | 0.5970     | 0.6029
3      | squared_error   | 0.6396     | 0.7714
3      | friedman_mse    | 0.6396     | 0.7714


C:\Users\May Zune\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\tree\_classes.py:1351: FutureWarning: Value `"friedman_mse"` for `criterion` is deprecated and will be removed in 1.11. It maps to `"squared_error"` as both were always equivalent. Use `criterion="squared_error"` to remove this warning.
  warnings.warn(


3      | absolute_error  | 0.6437     | 0.7176
3      | poisson         | 0.6294     | 0.7177
4      | squared_error   | 0.6376     | 0.7823
4      | friedman_mse    | 0.6376     | 0.7823


...

5      | absolute_error  | 0.7374     | 0.8257
5      | poisson         | 0.6487     | 0.7974
7      | squared_error   | 0.7410     | 0.7990


C:\Users\May Zune\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\tree\_classes.py:1351: FutureWarning: Value `"friedman_mse"` for `criterion` is deprecated and will be removed in 1.11. It maps to `"squared_error"` as both were always equivalent. Use `criterion="squared_error"` to remove this warning.
  warnings.warn(


7      | friedman_mse    | 0.7410     | 0.7990
7      | absolute_error  | 0.7033     | 0.7879
7      | poisson         | 0.6802     | 0.7664
None   | squared_error   | 0.6972     | 0.7553


C:\Users\May Zune\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\tree\_classes.py:1351: FutureWarning: Value `"friedman_mse"` for `criterion` is deprecated and will be removed in 1.11. It maps to `"squared_error"` as both were always equivalent. Use `criterion="squared_error"` to remove this warning.
  warnings.warn(


None   | friedman_mse    | 0.6972     | 0.7553
None   | absolute_error  | 0.7190     | 0.7987
None   | poisson         | 0.6723     | 0.7936

Best combination by CV Score:
Depth                  7.0
Criterion    squared_error
CV Score          0.740981
R²                0.799012
Name: 16, dtype: object


# XGBRegressor

XGBoost (eXtreme Gradient Boosting) is an optimized, 
distributed machine learning library designed to implement gradient-boosted decision trees (GBDT).

In [22]:
n_est = [5, 10, 50, 100, 150, 200, 500, 1000, 1200]
result = []
for n in n_est:
    xgb =XGBRegressor(
                n_estimators=n,        
                learning_rate=0.1,       
                booster='gbtree',        
                max_depth=6,             
                min_child_weight=1,      
                gamma=0,                
                subsample=1,             
                colsample_bytree=1,     
                reg_alpha=0,             
                reg_lambda=1,           
                tree_method='auto',      
                n_jobs=-1,               
                random_state=42,         
                verbosity=0              
            )
    y_hat = xgb.fit(x_train, y_train).predict(x_test)
    cv_score = cv(xgb, x_train, y_train, cv = kf).mean()
    r_square = r2_score(y_test, y_hat)
    mse = mean_squared_error(y_test, y_hat)
    result.append({
        'n_estimators' : n,
        'CV_Score'     : round(cv_score,4),
        'R_Square'     : round(r_square,4),
        'Squared_Error': round((mse),4)
    })
dframe = pd.DataFrame(result).sort_values(by = 'CV_Score', ascending = False)

print('-'*51)
print('|                  Summary                        |')
print('-'*51)
print(dframe)
print('-' * 30)
print('|     Best Estimator         |')
print('-'*30)
dframe.iloc[0]

---------------------------------------------------
|                  Summary                        |
---------------------------------------------------
   n_estimators  CV_Score  R_Square  Squared_Error
4           150    0.8563    0.9150   6.520258e+08
8          1200    0.8561    0.9157   6.467889e+08
7          1000    0.8561    0.9157   6.467914e+08
6           500    0.8561    0.9157   6.466977e+08
5           200    0.8561    0.9155   6.483091e+08
3           100    0.8559    0.9143   6.576860e+08
2            50    0.8513    0.9098   6.918630e+08
1            10    0.6799    0.7368   2.018834e+09
0             5    0.4820    0.5280   3.620509e+09
------------------------------
|     Best Estimator         |
------------------------------


n_estimators     1.500000e+02
CV_Score         8.563000e-01
R_Square         9.150000e-01
Squared_Error    6.520258e+08
Name: 4, dtype: float64

<div class="alert-success">
The preliminary test ends. XGBRegressor CV_Score and R_Square are higher.
</div>

<div class="alert-info">
Restart the process with a new df for prediction and submission work.
</div>

## new df

In [23]:
train_data = pd.read_csv('train.csv')
test_data  = pd.read_csv('test.csv')

# Keep Id for submission, drop from features
test_ids = test_data['Id']

train_df = train_data.drop(columns='Id')
test_df  = test_data.drop(columns='Id')

# Separate target
y = train_df['SalePrice']
X_train = train_df.drop(columns='SalePrice')
X_test  = test_df.copy()

print('Train shape:', X_train.shape)
print('Test shape :', X_test.shape)

Train shape: (1460, 79)
Test shape : (1459, 79)


### data check

In [24]:
class NATreatment(BaseEstimator, TransformerMixin):
    # Handles missing values. Learns LotFrontage group means from training data.

    def fit(self, X, y=None):
        # Learn group means from training data only — avoids data leakage
        self.lot_frontage_means_ = X.groupby('Street')['LotFrontage'].mean()
        self.electrical_mode_    = X['Electrical'].mode()[0]
        return self

    def transform(self, X):
        X = X.copy()

        # Fill with zero
        X['MasVnrArea'] = X['MasVnrArea'].fillna(0)

        # Garage: flag missing, then fill year with -999
        X['has_garage']  = np.where(X['GarageYrBlt'].isnull(), 1, 0)
        X['GarageYrBlt'] = X['GarageYrBlt'].fillna(-999)

        # LotFrontage: fill with learned group means (falls back to overall mean)
        lot_map = X['Street'].map(self.lot_frontage_means_)
        X['LotFrontage'] = X['LotFrontage'].fillna(lot_map)
        
        # If still NaN (unseen Street value in test), fill with overall mean
        X['LotFrontage'] = X['LotFrontage'].fillna(self.lot_frontage_means_.mean())

        # Features that are genuinely absent → 'None' string
        none_cols = ['Alley', 'MasVnrType', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
                     'BsmtFinType1', 'BsmtFinType2', 'FireplaceQu',
                     'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
                     'PoolQC', 'Fence', 'MiscFeature']
        X[none_cols] = X[none_cols].fillna('None')

        # Electrical: fill with mode learned from training
        X['Electrical'] = X['Electrical'].fillna(self.electrical_mode_)

        return X

### data transform

In [25]:
class ScaleAndEncode(BaseEstimator, TransformerMixin):
    """Ordinal encoding, label encoding of categoricals, PCA on garage features."""

    def fit(self, X, y=None):
        # Fit label encoders on training data so test can use the same mapping
        X_temp = self._apply_ordinal(X.copy())
        cat_cols = X_temp.select_dtypes(exclude=['int', 'float']).columns.tolist()
        self.label_encoders_ = {}
        for col in cat_cols:
            le = LabelEncoder()
            le.fit(X_temp[col].astype(str))
            self.label_encoders_[col] = le

        # Fit scaler + PCA on garage columns
        garage_cols = X_temp[['GarageYrBlt', 'GarageQual', 'GarageCond', 'has_garage']]
        self.garage_scaler_ = StandardScaler()
        SG = self.garage_scaler_.fit_transform(garage_cols)
        self.garage_pca_ = PCA(n_components=1)
        self.garage_pca_.fit(SG)
        return self

    def transform(self, X):
        X = self._apply_ordinal(X.copy())

        # Label encode categoricals using fitted encoders
        for col, le in self.label_encoders_.items():
            if col in X.columns:
                # Handle unseen labels in test data gracefully
                X[col] = X[col].astype(str).apply(
                    lambda v: le.transform([v])[0] if v in le.classes_ else -1
                )

        # PCA on garage features
        garage_cols = X[['GarageYrBlt', 'GarageQual', 'GarageCond', 'has_garage']]
        SG = self.garage_scaler_.transform(garage_cols)
        X['pca_garage'] = self.garage_pca_.transform(SG)
        X.drop(columns=['GarageYrBlt', 'GarageQual', 'GarageCond', 'has_garage',
                        'PoolArea'], inplace=True)

        return X

    def _apply_ordinal(self, X):
        """Deterministic ordinal mappings — no fitting needed."""
        Map_QC  = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}
        Map_BM  = {'GLQ': 6, 'ALQ': 5, 'BLQ': 4, 'Rec': 3, 'LwQ': 2, 'Unf': 1, 'None': 0}
        Map_BME = {'Gd': 4, 'Av': 3, 'Mn': 2, 'No': 1, 'None': 0}
        Map_GF  = {'Fin': 3, 'RFn': 2, 'Unf': 1, 'None': 0}
        Map_LS  = {'Gtl': 2, 'Mod': 1, 'Sev': 0}
        Map_LtS = {'Reg': 3, 'IR1': 2, 'IR2': 1, 'IR3': 0}

        for col in ['ExterQual','ExterCond','BsmtQual','BsmtCond',
                    'HeatingQC','KitchenQual','FireplaceQu','GarageQual','GarageCond','PoolQC']:
            X[col] = X[col].map(Map_QC)

        X['BsmtFinType1'] = X['BsmtFinType1'].map(Map_BM)
        X['BsmtFinType2'] = X['BsmtFinType2'].map(Map_BM)
        X['BsmtExposure'] = X['BsmtExposure'].map(Map_BME)
        X['GarageFinish'] = X['GarageFinish'].map(Map_GF)
        X['LandSlope']    = X['LandSlope'].map(Map_LS)
        X['LotShape']     = X['LotShape'].map(Map_LtS)
        return X

In [26]:
# XGBRegressor

xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.1,
    booster='gbtree',
    max_depth=6,
    min_child_weight=1,
    gamma=0,
    subsample=1,
    colsample_bytree=1,
    reg_alpha=0,
    reg_lambda=1,
    n_jobs=-1,
    random_state=42,
    verbosity=0
)

In [27]:
# pipe.fit

pipe = Pipeline([
    ('clean',  NATreatment()),    
    ('encode', ScaleAndEncode()), 
    ('model',  xgb)
])

pipe.fit(X_train, y)
print('Pipeline trained successfully!')

Pipeline trained successfully!


In [28]:
# Training R² (sanity check)

train_preds = pipe.predict(X_train)
print(f'Train R²: {r2_score(y, train_preds):.4f}')

# Predict on test set
test_preds = pipe.predict(X_test)

# Save submission
submission = pd.DataFrame({'Id': test_ids, 'SalePrice': test_preds})

# submission.to_csv('submission.csv', index=False)
# print('submission.csv saved!')

submission.head()

Train R²: 1.0000


,Id,SalePrice
0,1461,125688.320312
1,1462,168073.171875
2,1463,187390.937500
3,1464,184345.484375
4,1465,198817.546875


# Compare with the Sample submission

In [29]:
sample_submission  = pd.read_csv('sample_submission.csv')

In [30]:
sample_submission.head(3)

,Id,SalePrice
0,1461,169277.052498
1,1462,187758.393989
2,1463,183583.683570


In [31]:
sample_submissionX = sample_submission.rename(columns={"SalePrice": "SalePrice_Sample"})

In [32]:
df_merged = submission.merge(sample_submissionX, on='Id')

In [33]:
df_merged.head(2)

,Id,SalePrice,SalePrice_Sample
0,1461,125688.320312,169277.052498
1,1462,168073.171875,187758.393989


In [34]:
df_merged["difference_percent"] = ((df_merged["SalePrice"] - df_merged["SalePrice_Sample"])/ df_merged["SalePrice"])*100

In [35]:
df_merged.head(2)

,Id,SalePrice,SalePrice_Sample,difference_percent
0,1461,125688.320312,169277.052498,-34.680018
1,1462,168073.171875,187758.393989,-11.712293


In [36]:
df_merged.describe()

,Id,SalePrice,SalePrice_Sample,difference_percent
count,1459.000000,1459.000000,1459.000000,1459.000000
mean,2190.000000,178922.093750,179183.918243,-15.275243
std,421.321334,76229.890625,16518.303051,42.083314
min,1461.000000,48607.972656,135751.318893,-255.454242
25%,1825.500000,127853.148438,168703.011202,-38.111181
50%,2190.000000,158289.875000,179208.665698,-13.078043
75%,2554.500000,211246.445312,186789.409363,15.559468
max,2919.000000,618591.937500,281643.976117,66.990619


<div class="alert-danger">
Prediction differences are as low as -255% to as high as +66%.
</div>

<div class="alert-warning">
XGB underestimates the sale prices
</div>